<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase25_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
from collections import Counter
import hashlib, json, csv, io, zipfile

PHASE24_NAME = "ICHI2027_Phase24_SHAREABLE.json"
CANDIDATES = [Path('/content')/PHASE24_NAME, Path('/mnt/data')/PHASE24_NAME, Path.cwd()/PHASE24_NAME]
phase24_path = next((p for p in CANDIDATES if p.exists()), None)
if phase24_path is None:
    try:
        from google.colab import files
        print('Upload the Phase 24 SHAREABLE JSON only.')
        uploaded = files.upload()
        if PHASE24_NAME not in uploaded: raise FileNotFoundError('Required Phase 24 JSON not uploaded')
        phase24_bytes = uploaded[PHASE24_NAME]
    except ImportError:
        raise FileNotFoundError('Place ICHI2027_Phase24_SHAREABLE.json alongside this notebook.')
else:
    phase24_bytes = phase24_path.read_bytes()
phase24 = json.loads(phase24_bytes)
PHASE24_SHA256 = hashlib.sha256(phase24_bytes).hexdigest()
assert phase24['phase'] == '24_selected_original_generic_producer_portability_gate'
assert phase24['selected_original_producer']['source_sha256'] == 'd448fb9a2fa378d63cb199223d7142db627b5d6def53d2507d96fab10808262c'
assert phase24['gates']['phase23_upstream_verified'] is True
assert phase24['gates']['selected_original_cell_hash_verified'] is True
assert phase24['selected_original_producer']['portable_static_without_new_domain_rules'] is False
assert phase24['execution']['attempted'] is False
assert phase24['gates']['real_MRI_original_generic_C_executed'] is False
print('PASS: supplied Phase 24 gate verified; sha256:', PHASE24_SHA256)

Upload the Phase 24 SHAREABLE JSON only.


Saving ICHI2027_Phase24_SHAREABLE.json to ICHI2027_Phase24_SHAREABLE.json
PASS: supplied Phase 24 gate verified; sha256: 1f3af39cbb4c0e11fc4ab39d6d2a75189b65b5176baee799203fa30e67551078


In [2]:
producer=phase24['selected_original_producer']
arm_registry = {
 'A': {'name':'Faithful standards-supported mapping','status':'NOT_EXECUTED','role':'Future comparator; retains fields natively supported by pinned OMOP/MI-CDM revision'},
 'B': {'name':'Value-only loss ablation','status':'NOT_EXECUTED','role':'Future controlled stress test; not the standard baseline'},
 'C': {'name':'Exact original generic TFL v15 producer','status':'NOT_EXECUTABLE_ON_MRI_AS_IS','role':'Preserve nonportability finding; do not invent imaging semantics or claim an MRI runtime benchmark'},
 'D': {'name':'Proposed imaging-specific TFL','status':'SYNTHETIC_CONTRACT_PROTOTYPE_ONLY','role':'Prototype invariants below; no ETL integration or clinical validation'}
}
assert producer['domain_specific_code_hit_categories'] == ['encounter']
assert producer['portable_static_without_new_domain_rules'] is False
print(json.dumps(arm_registry,indent=2))

{
  "A": {
    "name": "Faithful standards-supported mapping",
    "status": "NOT_EXECUTED",
    "role": "Future comparator; retains fields natively supported by pinned OMOP/MI-CDM revision"
  },
  "B": {
    "name": "Value-only loss ablation",
    "status": "NOT_EXECUTED",
    "role": "Future controlled stress test; not the standard baseline"
  },
  "C": {
    "name": "Exact original generic TFL v15 producer",
    "status": "NOT_EXECUTABLE_ON_MRI_AS_IS",
    "role": "Preserve nonportability finding; do not invent imaging semantics or claim an MRI runtime benchmark"
  },
  "D": {
    "name": "Proposed imaging-specific TFL",
    "status": "SYNTHETIC_CONTRACT_PROTOTYPE_ONLY",
    "role": "Prototype invariants below; no ETL integration or clinical validation"
  }
}


### 2. Explicitly synthetic MRI evidence objects
All labels, scans, model versions, units and TRACE labels below are **invented fixtures**, not assertions about ADNI exports or official vocabulary concepts. `study_ref` and `source_ref` are toy local references, **not attested DICOM UIDs**. We use mm³ only as an illustrative synthetic unit. The source ground truth is deliberately simple and self-contained.

In [3]:
def make_event(person, side, visit, vol, scan_suffix):
    return {
        'person_key':person, 'study_ref':f'SYN-STUDY-{person}-{visit}',
        'source_ref':f'SYN-IMG-{person}-{visit}-{scan_suffix}',
        'visit':visit, 'region':'hippocampus', 'laterality':side,
        'volume':vol, 'unit':'mm3', 'model_name':'SyntheticSegmenter',
        'model_version':'0.1-test', 'source_link_resolved':True,
        'trace_state':'PASS', 'trace_rule_version':'synthetic-rule-1',
        'trace_reference':'synthetic-calibration-1',
        'provenance_ref':f'SYN-PROV-{person}-{visit}-{scan_suffix}'
    }
source = [make_event(p,side,visit,vol,suffix)
    for p, shift in [('SYN-P001',0),('SYN-P002',150)]
    for visit, delta in [('baseline',0),('month12',-80)]
    for side,vol,suffix in [('L',3500,'L'),('R',3550,'R')]
    for vol in [vol+shift+delta]]
assert len(source)==8
assert len({e['source_ref'] for e in source})==8
print('Synthetic participants:',len({e['person_key'] for e in source}), '| synthetic events:',len(source))

Synthetic participants: 2 | synthetic events: 8


In [4]:
REQUIRED = ('person_key','study_ref','source_ref','visit','region','laterality','volume','unit',
            'model_name','model_version','source_link_resolved','trace_state',
            'trace_rule_version','trace_reference','provenance_ref')
def normal_mm3(obj):
    value, unit = obj.get('volume'), obj.get('unit')
    if type(value) not in (int,float) or isinstance(value,bool): return None
    if unit=='mm3': return float(value)
    if unit=='cm3': return float(value)*1000
    return None
def contract_errors(reference, observed):
    errors=[]
    if any(f not in observed or observed[f] is None for f in REQUIRED):
        errors.append('REQUIRED_FIELD_MISSING')
    if observed.get('source_ref') != reference['source_ref'] or not observed.get('source_link_resolved'):
        errors.append('SOURCE_LINK_LOSS')
    if observed.get('person_key') != reference['person_key'] or observed.get('study_ref') != reference['study_ref'] or observed.get('visit') != reference['visit']:
        errors.append('PAIRING_OR_IDENTITY_LOSS')
    if observed.get('region') != reference['region'] or observed.get('laterality') != reference['laterality']:
        errors.append('ANATOMY_LATERALITY_LOSS')
    a,b=normal_mm3(reference),normal_mm3(observed)
    if a is None or b is None: errors.append('UNIT_UNRESOLVED')
    elif abs(a-b)>max(1e-6,1e-6*abs(a)): errors.append('VALUE_UNIT_MISMATCH')
    if observed.get('model_name') != reference['model_name'] or observed.get('model_version') != reference['model_version'] or observed.get('provenance_ref') != reference['provenance_ref']:
        errors.append('MODEL_PROVENANCE_LOSS')
    if any(observed.get(f) != reference[f] for f in ('trace_state','trace_rule_version','trace_reference')):
        errors.append('TRACE_STATE_LOSS')
    return sorted(set(errors))
def audit_case(original, candidate):
    # Index by source_ref; duplicate IDs and missing original IDs are both detectable.
    src={o['source_ref']:o for o in original}
    grouped={}
    for o in candidate: grouped.setdefault(o.get('source_ref'),[]).append(o)
    out=[]
    for key, ref in src.items():
        hits=grouped.get(key,[])
        errs=[] if hits else ['SOURCE_EVENT_MISSING']
        if len(hits)>1: errs.append('DUPLICATE_SOURCE_REF')
        if hits: errs.extend(contract_errors(ref,hits[0]))
        out.append({'source_ref':key,'person_key':ref['person_key'],'errors':sorted(set(errs))})
    for extra in set(grouped)-set(src):
        out.append({'source_ref':str(extra),'person_key':'SYNTHETIC_ONLY','errors':['UNEXPECTED_SOURCE_REF']})
    return out
assert not any(r['errors'] for r in audit_case(source,deepcopy(source)))
print('PASS: clean synthetic source passes independent prototype contract')

PASS: clean synthetic source passes independent prototype contract


### 4. E0–E7 controlled-corruption suite
These are targeted synthetic fixtures with **prespecified expected alarms**. E7 has two simultaneous faults; it does not establish robustness to unseen error families. The original generic TFL producer is **never invoked**.

In [5]:
cases={'E0_clean':deepcopy(source)}
def altered(name, transform):
    rows=deepcopy(source); transform(rows); cases[name]=rows
altered('E1_laterality_swap', lambda rows: rows[0].update(laterality='R'))
altered('E2_valid_unit_conversion', lambda rows: rows[0].update(volume=rows[0]['volume']/1000,unit='cm3'))
altered('E2_missing_unit', lambda rows: rows[0].update(unit=None))
altered('E3_visit_collision', lambda rows: rows[0].update(visit='month12'))
altered('E4_missing_model_version', lambda rows: rows[0].update(model_version=None))
altered('E5_trace_stripped', lambda rows: rows[0].update(trace_state=None))
altered('E6_duplicate_reference', lambda rows: rows.append(deepcopy(rows[0])))
altered('E6_unresolved_link', lambda rows: rows[0].update(source_link_resolved=False))
altered('E7_combined_fault', lambda rows: rows[0].update(laterality='R',model_version=None))
expected={
 'E0_clean':set(), 'E1_laterality_swap':{'ANATOMY_LATERALITY_LOSS'},
 'E2_valid_unit_conversion':set(), 'E2_missing_unit':{'UNIT_UNRESOLVED'},
 'E3_visit_collision':{'PAIRING_OR_IDENTITY_LOSS'},
 'E4_missing_model_version':{'MODEL_PROVENANCE_LOSS'},
 'E5_trace_stripped':{'TRACE_STATE_LOSS'},
 'E6_duplicate_reference':{'DUPLICATE_SOURCE_REF'},
 'E6_unresolved_link':{'SOURCE_LINK_LOSS'},
 'E7_combined_fault':{'ANATOMY_LATERALITY_LOSS','MODEL_PROVENANCE_LOSS'}
}
results=[]
for name, rows in cases.items():
    report=audit_case(source,rows)
    seen={e for row in report for e in row['errors']}
    ok=expected[name].issubset(seen) and (seen==set() if name in ('E0_clean','E2_valid_unit_conversion') else True)
    results.append({'fixture':name,'expected_alarms':sorted(expected[name]),'observed_alarms':sorted(seen),'passed':ok})
    assert ok, f'{name}: expected {expected[name]}, observed {seen}'
for r in results: print(r['fixture'], 'PASS' if r['passed'] else 'FAIL', '|', ','.join(r['observed_alarms']) or 'no warnings')

E0_clean PASS | no warnings
E1_laterality_swap PASS | ANATOMY_LATERALITY_LOSS
E2_valid_unit_conversion PASS | no warnings
E2_missing_unit PASS | REQUIRED_FIELD_MISSING,UNIT_UNRESOLVED
E3_visit_collision PASS | PAIRING_OR_IDENTITY_LOSS
E4_missing_model_version PASS | MODEL_PROVENANCE_LOSS,REQUIRED_FIELD_MISSING
E5_trace_stripped PASS | REQUIRED_FIELD_MISSING,TRACE_STATE_LOSS
E6_duplicate_reference PASS | DUPLICATE_SOURCE_REF
E6_unresolved_link PASS | SOURCE_LINK_LOSS
E7_combined_fault PASS | ANATOMY_LATERALITY_LOSS,MODEL_PROVENANCE_LOSS,REQUIRED_FIELD_MISSING


### 5. Synthetic source-cohort reference and blocked target equivalence claim
The next function defines only **synthetic source eligibility**: both hemispheres at both visits, convertible units, resolvable source link, and an unambiguous synthetic scan reference. It does **not** execute a target OMOP or MI-CDM query; therefore any equivalence outcome remains **NOT_TESTED**.

In [6]:
def synthetic_source_eligible(events):
    eligible=[]
    persons={o.get('person_key') for o in events}
    for person in sorted(persons):
        subset=[o for o in events if o.get('person_key')==person]
        if any(not o.get('source_link_resolved') or normal_mm3(o) is None for o in subset): continue
        if len({o.get('source_ref') for o in subset}) != len(subset): continue
        pairs=Counter((o.get('visit'),o.get('laterality')) for o in subset if o.get('region')=='hippocampus')
        if all(pairs.get((visit,side),0)==1 for visit in ('baseline','month12') for side in ('L','R')):
            eligible.append(person)
    return eligible
synthetic_reference=synthetic_source_eligible(source)
assert synthetic_reference==['SYN-P001','SYN-P002']
assert synthetic_source_eligible(cases['E6_duplicate_reference'])==['SYN-P002']
print('Synthetic source eligible participants:',len(synthetic_reference))
print('OMOP / MI-CDM target cohort equivalence: NOT TESTED')

Synthetic source eligible participants: 2
OMOP / MI-CDM target cohort equivalence: NOT TESTED


### 6. Privacy-safe Phase 25 exports
Exports include fixture **names and counts**, not rows, participant identifiers or per-event audit records. The SHA-256 points to the uploaded Phase 24 bytes; prior-source attestation remains inherited, not freshly replicated. The source audit SHA hash is recorded exactly as supplied by Phase 24. `true_four_arm_comparison_completed` stays `false`.

In [7]:
OUT=Path('/content') if Path('/content').exists() else Path.cwd()
report={
 'project':phase24['project'],
 'phase':'25_generic_nonportability_disposition_synthetic_contract_gate',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Synthetic-only execution; aggregate/public-safe export; no ADNI records, synthetic patient IDs, or private paths.',
 'phase24_sha256':PHASE24_SHA256,
 'phase24_source_producer_sha256':producer['source_sha256'],
 'phase24_source_producer_file':producer['source_file'],
 'phase24_source_producer_cell_index':producer['cell_index'],
 'arm_registry':arm_registry,
 'synthetic_fixture_summary':[{'fixture':r['fixture'],'passed':r['passed'],'warning_types':r['observed_alarms']} for r in results],
 'synthetic_total_fixtures':len(results),
 'synthetic_fixtures_passing_declared_expectations':sum(r['passed'] for r in results),
 'synthetic_reference_person_count':len(synthetic_reference),
 'gates':{
  'phase24_json_verified':True, 'exact_original_generic_C_portable_to_MRI':False,
  'exact_original_generic_C_MRI_executed':False,
  'synthetic_prototype_contract_checks_passed':all(r['passed'] for r in results),
  'FHIR_R4_export_executed':False,'pyOMOP_ETL_executed':False,
  'official_MICDM_load_completed':False,'real_MRI_rows_processed':0,
  'exact_ADNI_units_attested':False,'DICOM_UID_semantics_attested':False,
  'Athena_mappings_approved':False, 'true_four_arm_comparison_completed':False,
  'OMOP_target_cohort_equivalence_tested':False},
 'next_stage_gate':'Freeze versioned imaging contract and official target schema; implement synthetic FHIR representation and faithful mapping; only compare Arms A/B/D when executable, recording original C as nonportable rather than substituting a rewritten producer.'
}
json_path=OUT/'ICHI2027_Phase25_SHAREABLE.json'
csv_path=OUT/'ICHI2027_Phase25_SYNTHETIC_FIXTURE_SUMMARY.csv'
json_path.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
with csv_path.open('w',newline='',encoding='utf-8') as f:
    wr=csv.writer(f);wr.writerow(['fixture','passed','observed_warning_types'])
    for r in results:wr.writerow([r['fixture'],r['passed'],';'.join(r['observed_alarms'])])
print('Wrote:',json_path.name,csv_path.name)
print('Phase 25 gate:', 'PASS (synthetic only)' if report['gates']['synthetic_prototype_contract_checks_passed'] else 'FAIL')
try:
    from google.colab import files
    # Uncomment to download the public-safe gate artifacts in Colab:
    # files.download(str(json_path)); files.download(str(csv_path))
except ImportError: pass

Wrote: ICHI2027_Phase25_SHAREABLE.json ICHI2027_Phase25_SYNTHETIC_FIXTURE_SUMMARY.csv
Phase 25 gate: PASS (synthetic only)


### Phase 26 entry criteria
1. Pin an actual **official MI-CDM revision and OMOP CDM version** and approve the field/vocabulary crosswalk before claiming an official load.
2. Implement **synthetic FHIR R4** `Patient`/`ImagingStudy`/`Observation`/`Device`/`Provenance` with real link resolution and replay/idempotence tests.
3. Implement an independently auditable **faithful Arm A** preserving all fields supported by the pinned target.
4. Integrate the imaging-specific checker into a real synthetic ETL event sidecar (**proposed Arm D**), while preserving the exact nonportability finding for original Arm C.
5. Define paired, known-ground-truth error cases and execute same-record A/B/D comparison **only after** the mappings and contract are operational. Real ADNI processing stays blocked until source rights and units/identifier semantics are confirmed.

**Interpretation:** this notebook's 10 fixture checks are development tests, not performance estimates or evidence of clinical benefit.